In [1]:
!pip install pandas
!pip install ipython-sql prettytable 

import prettytable

prettytable.DEFAULT = 'DEFAULT'

In [2]:
import pandas as pd
import sqlite3

In [3]:
census_df = pd.read_csv("ChicagoCensusData.csv")
schools_df = pd.read_csv("ChicagoPublicSchools.csv")
crime_df = pd.read_csv("ChicagoCrimeData.csv")

In [4]:
# This will create or connect to a SQLite database file named FinalDB.db
conn = sqlite3.connect("FinalDB.db")

In [5]:
# Store each dataframe as a table in SQLite
census_df.to_sql("CHICAGO_CENSUS_DATA", conn, if_exists='replace', index=False)
schools_df.to_sql("CHICAGO_PUBLIC_SCHOOLS", conn, if_exists='replace', index=False)
crime_df.to_sql("CHICAGO_CRIME_DATA", conn, if_exists='replace', index=False)

533

In [6]:
# List all tables in the database
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('CHICAGO_CENSUS_DATA',), ('CHICAGO_PUBLIC_SCHOOLS',), ('CHICAGO_CRIME_DATA',)]


In [7]:
%load_ext sql

In [8]:
%sql sqlite:///FinalDB.db

In [9]:
%%sql
SELECT * FROM CHICAGO_CENSUS_DATA LIMIT 5;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX
1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0


In [13]:
#Problem 1
%%sql
SELECT COUNT(*) AS TotalCrimes FROM CHICAGO_CRIME_DATA;

 * sqlite:///FinalDB.db
Done.


TotalCrimes
533


In [15]:
%%sql
SELECT COMMUNITY_AREA_NUMBER, COMMUNITY_AREA_NAME 
FROM CHICAGO_CENSUS_DATA 
WHERE PER_CAPITA_INCOME < 11000;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME
26.0,West Garfield Park
30.0,South Lawndale
37.0,Fuller Park
54.0,Riverdale


In [16]:
%%sql
SELECT CASE_NUMBER 
FROM CHICAGO_CRIME_DATA 
WHERE DESCRIPTION LIKE '%MINOR%';

 * sqlite:///FinalDB.db
Done.


CASE_NUMBER
HL266884
HK238408


In [17]:
%%sql
SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY 
FROM CHICAGO_CENSUS_DATA 
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC 
LIMIT 5;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4


In [18]:
%%sql
SELECT DISTINCT PRIMARY_TYPE 
FROM CHICAGO_CRIME_DATA 
WHERE LOCATION_DESCRIPTION LIKE '%SCHOOL%';

 * sqlite:///FinalDB.db
Done.


PRIMARY_TYPE
BATTERY
CRIMINAL DAMAGE
NARCOTICS
ASSAULT
CRIMINAL TRESPASS
PUBLIC PEACE VIOLATION


In [19]:
%%sql
SELECT "Elementary, Middle, or High School" AS School_Type, 
       AVG(SAFETY_SCORE) AS Average_Safety_Score 
FROM CHICAGO_PUBLIC_SCHOOLS 
GROUP BY "Elementary, Middle, or High School";

 * sqlite:///FinalDB.db
Done.


School_Type,Average_Safety_Score
ES,49.52038369304557
HS,49.62352941176471
MS,48.0


In [20]:
%%sql
SELECT COMMUNITY_AREA_NUMBER 
FROM CHICAGO_CRIME_DATA 
GROUP BY COMMUNITY_AREA_NUMBER 
ORDER BY COUNT(*) DESC 
LIMIT 1;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER
25.0


In [22]:
%%sql
SELECT COMMUNITY_AREA_NAME 
FROM CHICAGO_CENSUS_DATA 
WHERE COMMUNITY_AREA_NUMBER = (
    SELECT COMMUNITY_AREA_NUMBER 
    FROM CHICAGO_CRIME_DATA 
    GROUP BY COMMUNITY_AREA_NUMBER 
    ORDER BY COUNT(*) DESC 
    LIMIT 1
);

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Austin
